# RAG — Stage 4: Augmented generation (JDECo actual run)

**Goal:** run a real retrieval-augmented generation pipeline on the uploaded JDECo SRS file.

**Pipeline:** load → chunk → index → `retrieve_top_k` → `answer_with_context`.

**Requirements:** `OPENAI_API_KEY` in `rag_lab/.env`.

**Output in this notebook:**
- single-question demo (retrieval + answer),
- multi-question run with actual chunk IDs and generated answers,
- CSV export for paper evidence.


In [ ]:
%pip install -q langchain-text-splitters langchain-core langchain-community langchain-openai faiss-cpu pymupdf openai python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import sys
from pathlib import Path

# Run this notebook from the notebooks/ folder (parent = rag_lab).
RAG_LAB = Path("..").resolve()
SRC = RAG_LAB / "src"
sys.path.insert(0, str(SRC))

from qbrain_rag.application.chunking import chunk_text
from qbrain_rag.infrastructure.vector_store import retrieve_top_k
from qbrain_rag.config.notebook_defaults import resolve_default_srs_path
from qbrain_rag.infrastructure.document_loaders import load_document
from qbrain_rag.infrastructure.llm import answer_with_context
from qbrain_rag.infrastructure.vector_store import build_faiss_store

srs_path = resolve_default_srs_path(RAG_LAB)
text = load_document(str(srs_path))
chunks = chunk_text(text)
metadata = [{"source_file": srs_path.name, "chunk_id": i + 1} for i in range(len(chunks))]
store = build_faiss_store(chunks, metadata)

question = "What should the system display while a function is being processed?"
docs = retrieve_top_k(store, question, k=5)
answer = answer_with_context(question, docs)

print("=== Question ===\n", question)
print("\n=== Answer ===\n")
print(answer)


=== Question ===
 What should the system display while a function is being processed?

=== Answer ===

The system shall display an icon indicating the processing when a function is being processed (UL-5).


## Multi-question actual RAG run (for paper evidence)

This cell runs several real questions against the JDECo index, then saves a CSV with:
- question,
- retrieved chunk IDs,
- retrieved source file,
- generated answer.

In [ ]:
import pandas as pd

questions = [
    "How can a customer submit a service request in JDECo-SMS?",
    "What should the system display while a function is being processed?",
    "Who can access all service requests according to the SRS?",
    "What happens if required fields are missing during request submission?",
    "How is payment request handling described in the workflow?",
]

k = 5
rows = []

for q in questions:
    docs_q = retrieve_top_k(store, q, k=k)
    ans_q = answer_with_context(q, docs_q, temperature=0.1, evaluation_mode=True)
    chunk_ids = [d.metadata.get("chunk_id") for d in docs_q]
    src_files = sorted({str(d.metadata.get("source_file", "")) for d in docs_q})
    preview = " ".join(docs_q[0].page_content.split())[:220] + "..." if docs_q else ""

    rows.append(
        {
            "question": q,
            "k": k,
            "retrieved_chunk_ids": chunk_ids,
            "retrieved_sources": ";".join(src_files),
            "top1_preview": preview,
            "generated_answer": ans_q,
        }
    )

results_df = pd.DataFrame(rows)
display(results_df[["question", "retrieved_chunk_ids", "generated_answer"]])

out_csv = RAG_LAB / "results" / "paper_output_sample" / "jdeco_actual_rag_outputs.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print("Saved:", out_csv)

,question,retrieved_chunk_ids,generated_answer
0,How can a customer submit a service request in...,"[21, 14, 13, 6, 12]",The customer can submit a service request by a...
1,What should the system display while a functio...,"[23, 22, 24, 17, 4]",The system shall display an icon indicating th...
2,Who can access all service requests according ...,"[20, 22, 21, 27, 25]",CS (Customer Services) can access all service ...
3,What happens if required fields are missing du...,"[22, 26, 20, 28, 14]",JDECo-SMS validates the data entered and notif...
4,How is payment request handling described in t...,"[9, 8, 10, 17, 15]",The payment request is generated by the Financ...


Saved: D:\Qbrainpython\QBrain\rag_lab\results\paper_output_sample\jdeco_actual_rag_outputs.csv
